In [1]:
# import libraries
import polars as pl
from datetime import date
from tqdm.notebook import tqdm
import plotly.express as px

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_width_chars(200)

# data path
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.config import RAW_DIR, INTERIM_DIR, PROCESSED_DIR, DOCS_DIR

# 3. Data Preparation

## 3.1. Objective

Transform the raw dataset into a clean, consistent, leakage-controlled dataset suitable for downstream PD modeling.

The preparation follows the decisions established during Data Understanding:

* Analysis period starts January 2016.
* Only resolved loans are retained.
* Target is defined as Paid vs Default.
* Post-origination and target-related features are excluded.
* High-missingness features are removed.
* Variables are converted to appropriate analytical data types.

## 3.2. Load Data

In [2]:
# load data
lf = pl.scan_csv(
    RAW_DIR / "lending_club/Loan_status_2007-2020Q3.gzip",
    infer_schema_length=10000,
    ignore_errors=True)

In [3]:
lf.collect_schema()

Schema([('', Int64),
        ('id', Int64),
        ('loan_amnt', Int64),
        ('funded_amnt', Int64),
        ('funded_amnt_inv', Float64),
        ('term', String),
        ('int_rate', String),
        ('installment', Float64),
        ('grade', String),
        ('sub_grade', String),
        ('emp_title', String),
        ('emp_length', String),
        ('home_ownership', String),
        ('annual_inc', Float64),
        ('verification_status', String),
        ('issue_d', String),
        ('loan_status', String),
        ('pymnt_plan', String),
        ('url', String),
        ('purpose', String),
        ('title', String),
        ('zip_code', String),
        ('addr_state', String),
        ('dti', Float64),
        ('delinq_2yrs', Int64),
        ('earliest_cr_line', String),
        ('fico_range_low', Int64),
        ('fico_range_high', Int64),
        ('inq_last_6mths', Int64),
        ('mths_since_last_delinq', Int64),
        ('mths_since_last_record', Int64),
        ('

In [4]:
lf.head(5).collect()

,id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,debt_settlement_flag
i64,i64,i64,i64,f64,str,str,f64,str,str,str,str,str,f64,str,str,str,str,str,str,str,str,str,f64,i64,str,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,i64,i64,i64,str,i64,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
0,1077501,5000,5000,4975.0,""" 36 months""",""" 10.65%""",162.87,"""B""","""B2""",null,"""10+ years""","""RENT""",24000.0,"""Verified""","""Dec-2011""","""Fully Paid""","""n""","""https://lendingclub.com/browse/loanDetail.action?loan_id=1077501""","""credit_card""","""Computer""","""860xx""","""AZ""",27.65,0,"""Jan-1985""",735,739,1,null,null,3,0,13648,"""83.7%""",9,"""f""",0.0,0.0,5863.155187,5833.84,5000.0,863.16,0.0,0.0,0.0,"""Jan-2015""",171.62,null,"""May-2020""",704,700,0,null,1,"""Individual""",null,null,null,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""N""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""N"""
1,1077430,2500,2500,2500.0,""" 60 months""",""" 15.27%""",59.83,"""C""","""C4""","""Ryder""","""< 1 year""","""RENT""",30000.0,"""Source Verified""","""Dec-2011""","""Charged Off""","""n""","""https://lendingclub.com/browse/loanDetail.action?loan_id=1077430""","""car"

## 3.3. Filter Data

In [5]:
# filter date
df = (lf
        .with_columns(pl.col("issue_d").str.strptime(pl.Date, "%b-%Y", strict=False).alias("issue_d"))  # cast date
        .filter(pl.col("issue_d") >= date(2016, 1, 1))                                                  # filter out data before 2016
    )

In [6]:
df.select(pl.len()).collect()

len
u32
2038052


In [7]:
# filter loan status
LOAN_STATUS_MAP = {
    "Charged Off" : "default",
    "Default" : "default",
    "Does not meet the credit policy. Status:Charged Off" : "default",
    "Fully Paid" : "paid",
    "Does not meet the credit policy. Status:Fully Paid" : "paid",
    "Current": "not resolve",
    "In Grace Period": "not resolve",
    "Late (16-30 days)": "not resolve",
    "Late (31-120 days)": "not resolve",
    "Issued" : "not resolve"
}

df = (df
      .with_columns(
          pl.col("loan_status")
          .replace(LOAN_STATUS_MAP)                             # simplify loan status
          .alias("loan_status")
          .cast(pl.String))    
      .filter(pl.col("loan_status").is_in(["default", "paid"])) # filter resolved loan status
)   

In [8]:
df.select(pl.len()).collect()

len
u32
994341


## 3.4. Drop Columns

In [9]:
# count column
lf.collect_schema().len()

142

In [10]:
# load data profile
data_profile = pl.read_parquet(INTERIM_DIR / "data_profile.parquet")

In [11]:
# filter pd eligible features
# list pd eligible features
col_pd_eligible = (data_profile
                    .filter(pl.col("pd_eligible") == True)   # keep only eligible rows
                    .select(pl.col("feature"))
                    .to_series()
                    .to_list())

col_pd_eligible = col_pd_eligible + ["loan_status", "issue_d"]  # add target column

# filter columns
df = df.select(col_pd_eligible)

In [12]:
df.collect_schema().len()

104

In [13]:
# drop null percentage > 90%
# list all columns
df_col = df.collect_schema().names()

# calculate null percentage
null_pct = (
    df
    .select([
        pl.col(col)
        .is_null()
        .mean()
        .mul(100)
        .round(2)
        .alias(col)
        for col in df_col
    ])
    .unpivot(
        variable_name="feature",
        value_name="missing_pct",
    )
    .sort("missing_pct", descending=True)
    .collect()
)

# list features with missing percentage > 90%
col_null = null_pct.filter(pl.col("missing_pct") > 90).select(pl.col("feature")).to_series().to_list()

# filter columns
df = df.select(pl.exclude(col_null))

In [14]:
df.collect_schema().len()

89

In [15]:
# filter post-origination features
# list post-origination features
col_post_origin = (data_profile
                    .filter(pl.col("availability") == "post_origination")   # keep only eligible rows
                    .select(pl.col("feature"))
                    .to_series()
                    .to_list())

# filter columns
df = df.select(pl.exclude(col_post_origin))

In [16]:
df.collect_schema().len()

89

## 3.5. Data Type Normalization

In [17]:
df.head(5).collect()

loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,purpose,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,collections_12_mths_ex_med,mths_since_last_major_derog,application_type,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,loan_status,issue_d
i64,i64,f64,str,str,f64,str,str,str,str,f64,str,str,str,str,f64,i64,str,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,str,str,str,str,date
12000,12000,12000.0,""" 36 months""",""" 7.97%""",375.88,"""A""","""A5""","""10+ years""","""OWN""",42000.0,"""Source Verified""","""debt_consolidation""","""923xx""","""CA""",27.74,0,"""Jun-1996""",715,719,0,null,80,9,1,11457,"""37%""",16,"""w""",0,null,"""Individual""",0,"""0""","""30502""","""1""","""2""","""1""","""3""","""8""","""19045""","""73""","""2""","""4""","""7117""","""53""","""31000""","""1""","""1""","""2""","""7""","""3389""","""7144""","""53.9""",0,0,"""131""","""255""","""1""","""1""","""0""","""14""",null,"""8""",null,"""0""","""2""","""6""","""2""","""2""","""7""","""7""","""9""","""6""","""9""","""0""","""0""","""0""","""3""","""100""","""0""",1,0,"""57180""","""30502""","""15500""","""26180""","""paid""",2017-09-01
10000,10000,10000.0,""" 36 months""",""" 9.44%""",320.05,"""B""","""B1""","""3 years""","""MORTGAGE""",55000.0,"""Not Verified""","""debt_consolidation""","""971xx""","""OR""",18.79,0,"""Sep-2005""",695,699,0,68,99,7,1,7188,"""57.5%""",10,"""w""",0,null,"""Individual""",0,"""0""","""340607""","""0""","""4""","""1""","""2""","""10""","""209187""","""75""","""0""","""0""","""6847""","""68""","""12500""","""1""","""1""","""1""","""3""","""48658""","""3653""","""65.2""",0,0,"""144""","""73""","""49""","""10""","""2""","""49""",null,"""8""","""68""","""0""","""1""","""2""","""1""","""1""","""5""","""2""","""3""","""2""","""7""","""0""","""0""","""0""","""1""","""100""","""0""",1,0,"""315034""","""216375""","""10500""","""175048""","""paid""",2017-09-01
8000,8000,8000.0,""" 36 months""",""" 16.02%""",281.34,"""C""","""C5""","""< 1 year""","""MORTGAGE""",120000.0,"""Not Verified""","""debt_consolidation""","""136xx""","""NY""",20.36,0,"""Sep-1994""",700,704,1,64,null,8,0,19015,"""92.3%""",34,"""w""",0,null,"""Joint App""",0,"""0""","""388595""","""2""","""3""","""2""","""5""","""6""","""83572""","""85""","""0""","""0""","""0""","""88""","""20600""","""1""","""24""","""2""","""6""","""55514""",null,null,0,0,"""137""","""276""","""34""","""6""","""6""","""152""",null,"""3""",null,"""0""","""0""","""3""","""1""","""2""","""19""","""4""","""9""","""3""","""8""","""0""","""0""","""0""","""3""","""97.1""",null,0,0,"""400558""","""102587""","""0""","""89958""","""paid""",2017-09-01
12800,12800,12800.0,""" 36 months""",""" 13.59%""",434.93,"""C""","""C2""","""5 years""","""RENT""",90000

In [18]:
# cast data type
df = df.with_columns([

    # cast term to int
    pl.col("term")
      .str.strip_chars()
      .str.replace(" months", "")
      .cast(pl.Int8, strict=False),

    # cast int_rate
    pl.col("int_rate")
      .str.strip_chars()
      .str.replace("%", "")
      .cast(pl.Float64, strict=False),

    # cast revol_util
    pl.col("revol_util")
      .str.strip_chars()
      .str.replace("%", "")
      .cast(pl.Float64, strict=False),

    # cast emp_length
    pl.col("emp_length")
      .str.strip_chars()
      .str.replace(" years", "")
      .str.replace(" year", "")
      .str.replace("< 1", "0")
      .str.replace("10+", "10")
      .cast(pl.Int8, strict=False)
])

In [19]:
df.head(5).collect()

loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,purpose,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,collections_12_mths_ex_med,mths_since_last_major_derog,application_type,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,loan_status,issue_d
i64,i64,f64,i8,f64,f64,str,str,i8,str,f64,str,str,str,str,f64,i64,str,i64,i64,i64,i64,i64,i64,i64,i64,f64,i64,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,str,str,str,str,date
12000,12000,12000.0,36,7.97,375.88,"""A""","""A5""",null,"""OWN""",42000.0,"""Source Verified""","""debt_consolidation""","""923xx""","""CA""",27.74,0,"""Jun-1996""",715,719,0,null,80,9,1,11457,37.0,16,"""w""",0,null,"""Individual""",0,"""0""","""30502""","""1""","""2""","""1""","""3""","""8""","""19045""","""73""","""2""","""4""","""7117""","""53""","""31000""","""1""","""1""","""2""","""7""","""3389""","""7144""","""53.9""",0,0,"""131""","""255""","""1""","""1""","""0""","""14""",null,"""8""",null,"""0""","""2""","""6""","""2""","""2""","""7""","""7""","""9""","""6""","""9""","""0""","""0""","""0""","""3""","""100""","""0""",1,0,"""57180""","""30502""","""15500""","""26180""","""paid""",2017-09-01
10000,10000,10000.0,36,9.44,320.05,"""B""","""B1""",3,"""MORTGAGE""",55000.0,"""Not Verified""","""debt_consolidation""","""971xx""","""OR""",18.79,0,"""Sep-2005""",695,699,0,68,99,7,1,7188,57.5,10,"""w""",0,null,"""Individual""",0,"""0""","""340607""","""0""","""4""","""1""","""2""","""10""","""209187""","""75""","""0""","""0""","""6847""","""68""","""12500""","""1""","""1""","""1""","""3""","""48658""","""3653""","""65.2""",0,0,"""144""","""73""","""49""","""10""","""2""","""49""",null,"""8""","""68""","""0""","""1""","""2""","""1""","""1""","""5""","""2""","""3""","""2""","""7""","""0""","""0""","""0""","""1""","""100""","""0""",1,0,"""315034""","""216375""","""10500""","""175048""","""paid""",2017-09-01
8000,8000,8000.0,36,16.02,281.34,"""C""","""C5""",0,"""MORTGAGE""",120000.0,"""Not Verified""","""debt_consolidation""","""136xx""","""NY""",20.36,0,"""Sep-1994""",700,704,1,64,null,8,0,19015,92.3,34,"""w""",0,null,"""Joint App""",0,"""0""","""388595""","""2""","""3""","""2""","""5""","""6""","""83572""","""85""","""0""","""0""","""0""","""88""","""20600""","""1""","""24""","""2""","""6""","""55514""",null,null,0,0,"""137""","""276""","""34""","""6""","""6""","""152""",null,"""3""",null,"""0""","""0""","""3""","""1""","""2""","""19""","""4""","""9""","""3""","""8""","""0""","""0""","""0""","""3""","""97.1""",null,0,0,"""400558""","""102587""","""0""","""89958""","""paid""",2017-09-01
12800,12800,12800.0,36,13.59,434.93,"""C""","""C2""",5,"""RENT""",90000.0,"""Not Verified""","""debt_consolidation""","""799xx""","""TX""",22.63,0,"""Dec-1988""",660,664,2,43,null,10,0,12660,86.0,23,"""w""",0,"""94""","""Individ

In [20]:
# current data feature
data_profile_now = data_profile.filter(pl.col("feature").is_in(df.collect_schema().names()))

# dtype mapping
dtype_map = {
    "Int64": pl.Int64,
    "Float64": pl.Float64,
    "string": pl.String,
    "category": pl.Categorical,
    "datetime64[ns]": pl.Date,
}

# define feature and their cast
feature_list = data_profile_now["feature"].to_list()

dtype_lists = data_profile_now["cast"].to_list()
dtype_lists = [dtype_map[i] for i in dtype_lists]

# cast
cast_exec = [pl.col(column).cast(dtype, strict=False)
            for column, dtype in zip(feature_list, dtype_lists)
            if column != "earliest_cr_line"]

cast_exec = cast_exec + [pl.col("earliest_cr_line").str.strptime(pl.Date, "%b-%Y", strict=False)]

df = df.with_columns(cast_exec)

In [21]:
df.head(5).collect()

loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,purpose,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,collections_12_mths_ex_med,mths_since_last_major_derog,application_type,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,loan_status,issue_d
i64,i64,f64,i64,f64,f64,cat,cat,i64,cat,f64,cat,cat,str,cat,f64,i64,date,i64,i64,i64,i64,i64,i64,i64,f64,f64,i64,cat,i64,i64,cat,i64,f64,f64,i64,i64,i64,i64,i64,f64,f64,i64,i64,f64,f64,f64,i64,i64,i64,i64,f64,f64,f64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,i64,i64,f64,f64,f64,f64,cat,date
12000,12000,12000.0,36,7.97,375.88,"""A""","""A5""",null,"""OWN""",42000.0,"""Source Verified""","""debt_consolidation""","""923xx""","""CA""",27.74,0,1996-06-01,715,719,0,null,80,9,1,11457.0,37.0,16,"""w""",0,null,"""Individual""",0,0.0,30502.0,1,2,1,3,8,19045.0,73.0,2,4,7117.0,53.0,31000.0,1,1,2,7,3389.0,7144.0,53.9,0,0.0,131,255,1,1,0,14,null,8,null,0,2,6,2,2,7,7,9,6,9,0,0,0,3,100.0,0.0,1,0,57180.0,30502.0,15500.0,26180.0,"""paid""",2017-09-01
10000,10000,10000.0,36,9.44,320.05,"""B""","""B1""",3,"""MORTGAGE""",55000.0,"""Not Verified""","""debt_consolidation""","""971xx""","""OR""",18.79,0,2005-09-01,695,699,0,68,99,7,1,7188.0,57.5,10,"""w""",0,null,"""Individual""",0,0.0,340607.0,0,4,1,2,10,209187.0,75.0,0,0,6847.0,68.0,12500.0,1,1,1,3,48658.0,3653.0,65.2,0,0.0,144,73,49,10,2,49,null,8,68,0,1,2,1,1,5,2,3,2,7,0,0,0,1,100.0,0.0,1,0,315034.0,216375.0,10500.0,175048.0,"""paid""",2017-09-01
8000,8000,8000.0,36,16.02,281.34,"""C""","""C5""",0,"""MORTGAGE""",120000.0,"""Not Verified""","""debt_consolidation""","""136xx""","""NY""",20.36,0,1994-09-01,700,704,1,64,null,8,0,19015.0,92.3,34,"""w""",0,null,"""Joint App""",0,0.0,388595.0,2,3,2,5,6,83572.0,85.0,0,0,0.0,88.0,20600.0,1,24,2,6,55514.0,null,null,0,0.0,137,276,34,6,6,152,null,3,null,0,0,3,1,2,19,4,9,3,8,0,0,0,3,97.1,null,0,0,400558.0,102587.0,0.0,89958.0,"""paid""",2017-09-01
12800,12800,12800.0,36,13.59,434.93,"""C""","""C2""",5,"""RENT""",90000.0,"""Not Verified""","""debt_consolidation""","""799xx""","""TX""",22.63,0,1988-12-01,660,664,2,43,null,10,0,12660.0,86.0,23,"""w""",0,94,"""Individual""",0,0.0,93375.0,2,3,2,3,6,80715.0,91.0,1,3,3777.0,86.0,14750.0,0,1,2,6,10375.0,2458.0,86.0,0,0.0,154,345,5,5,0,5,null,5,null,0,6,6,6,6,15,6,8,6,9,null,0,0,3,83.0,83.3,0,0,103040.0,93375.0,14750.0,88290.0,"""paid""",2017-09-01
15000,15000,15000.0,36,13.59,509.69,"""C""","""C2""",4,"""MORTGAGE""",180000.0,"""Source Verified""","""medical""","""757xx""","""TX""",38.07,0,1976-12-01,680,684,0,null,null,24,0,107214.0,66.6,50,"""w""",0,null,"""Individual""",0,0.0,682000.0,0,4,0,3,13,190233.0,79.0,0,1,22127.0,68.0,161100.0,2,8,0,5,29652.0,16347.0,86.0,0,0.0,132,489,16,13,4,16,null,13,null,0,10,14,11,16,17,19,29,14,24,0,0,0,0,100.0,72.7,0,0,769704.0,297447.0,116700.0,221207.0,"""paid""",2017-09-01


## 3.6. Data Quality Check

In [22]:
def create_data_profile(
    lf: pl.LazyFrame,
    description_df: pl.DataFrame,
    unique_sample_size: int = 5,
) -> pl.DataFrame:
    """
    Create a column-level data profile from a Polars LazyFrame.

    Includes:
    - feature
    - description
    - dtype
    - count
    - null_count
    - null_pct
    - unique_count
    - unique_sample
    - mean
    - median
    - mode

    The description is retrieved from a CSV containing:
    - LoanStatNew
    - Description

    If a feature does not have a matching description,
    the description is returned as null.

    Mean and median are only calculated for numeric columns.
    """

    # ---------------------------------------------------------
    # 1. Load feature descriptions
    # ---------------------------------------------------------

    description_df = (
        description_df
        .select([
            pl.col("LoanStatNew").alias("feature"),
            pl.col("Description").alias("description"),
        ])
        .with_columns(
            pl.col("feature").cast(pl.String),
            pl.col("description").cast(pl.String),
        )
        .unique(subset=["feature"])
    )

    # ---------------------------------------------------------
    # 2. Get schema
    # ---------------------------------------------------------

    schema = lf.collect_schema()

    profile = []

    # ---------------------------------------------------------
    # 3. Profile each feature
    # ---------------------------------------------------------

    for col, dtype in tqdm(schema.items()):

        stats = (
            lf.select([
                pl.len().alias("count"),
                pl.col(col).null_count().alias("null_count"),
                pl.col(col).n_unique().alias("unique_count"),
            ])
            .collect()
            .row(0)
        )

        count, null_count, unique_count = stats

        null_pct = (
            null_count / count * 100
            if count > 0
            else 0
        )

        # -----------------------------------------------------
        # Unique sample
        # -----------------------------------------------------

        unique_sample = (
            lf.select(
                pl.col(col)
                .drop_nulls()
                .unique()
                .head(unique_sample_size)
                .alias(col)
            )
            .collect()
            .get_column(col)
            .to_list()
        )

        # -----------------------------------------------------
        # Default statistics
        # -----------------------------------------------------

        mean = None
        median = None
        mode = None

        # -----------------------------------------------------
        # Numeric statistics
        # -----------------------------------------------------

        if dtype.is_numeric():

            numeric_stats = (
                lf.select([
                    pl.col(col).mean().alias("mean"),
                    pl.col(col).median().alias("median"),
                ])
                .collect()
                .row(0)
            )

            mean, median = numeric_stats

        # -----------------------------------------------------
        # Mode
        # -----------------------------------------------------

        mode_result = (
            lf.select(
                pl.col(col)
                .drop_nulls()
                .mode()
                .head(1)
                .alias("mode")
            )
            .collect()
            .get_column("mode")
            .to_list()
        )

        if mode_result:
            mode = mode_result[0]

        # -----------------------------------------------------
        # Store profile
        # -----------------------------------------------------

        profile.append({
            "feature": col,
            "dtype": str(dtype),
            "count": count,
            "null_count": null_count,
            "null_pct": null_pct,
            "unique_count": unique_count,
            "unique_sample": str(unique_sample),
            "mean": str(mean),
            "median": str(median),
            "mode": str(mode),
        })

    # ---------------------------------------------------------
    # 4. Create profile DataFrame
    # ---------------------------------------------------------

    profile_df = pl.DataFrame(profile)

    # ---------------------------------------------------------
    # 5. Add descriptions
    # ---------------------------------------------------------

    profile_df = (
        profile_df
        .join(
            description_df,
            on="feature",
            how="left",
        )
        .select([
            "feature",
            "description",
            "dtype",
            "count",
            "null_count",
            "null_pct",
            "unique_count",
            "unique_sample",
            "mean",
            "median",
            "mode",
        ])
    )

    return profile_df

In [23]:
# # create data profile from data clean
# description_df = pl.read_excel(RAW_DIR / "lending_club/LCDataDictionary.xlsx")
# data_profile = create_data_profile(df, description_df)

# # save data profile to parquet
# data_profile.write_parquet(INTERIM_DIR / "data_profile_model_base.parquet")

# load data profile
data_profile = pl.read_parquet(INTERIM_DIR / "data_profile_model_base.parquet")

In [24]:
# prettify data display
from great_tables import GT
from IPython.display import HTML, display

table = (GT(data_profile).tab_header(title="Dataset Profile Summary"))

html = table._repr_html_()

display(HTML(f"""
<div style="
    max-height: 600px;
    overflow-y: auto;
    overflow-x: auto;
    border: 1px solid #ddd;
">
    {html}
</div>
"""))

## 3.7. Save File

In [25]:
df.sink_parquet(PROCESSED_DIR / "modelling_base.parquet")

## 3.8. Summary

In [46]:
# preparation summary

# define temp variable
lf_miss = lf.select(pl.sum_horizontal(pl.all().null_count())).collect().item()
lf_cell = lf.select(pl.len()).collect().item() * len(lf.collect_schema())

df_miss = df.select(pl.sum_horizontal(pl.all().null_count())).collect().item()
df_cell = df.select(pl.len()).collect().item() * len(df.collect_schema())

# summary
summary = pl.DataFrame({
    "metric": [
        "Rows",
        "Columns",
        "Date range",
        "Default rate",
        "Missing cells",
        "Missing cell rate",
    ],
    "before": [
        lf.select(pl.len()).collect().item(),
        len(lf.collect_schema()),
        str(lf.select(pl.col("issue_d").str.strptime(pl.Date, "%b-%Y", strict=False).min()).collect().item()),
        lf.select((pl.col("loan_status")
                   .is_in(["Charged Off", "Default", "Does not meet the credit policy. Status:Charged Off"]))
                   .cast(pl.Float64).mean().round(4) * 100.0).collect().item(),
        lf_miss,
        round((lf_miss / lf_cell) * 100.0, 2),
    ],
    "after": [
        df.select(pl.len()).collect().item(),
        len(df.collect_schema()),
        str(df.select(pl.col("issue_d").min()).collect().item()),
        df.select((pl.col("loan_status")=="default").cast(pl.Float64).mean().round(4) * 100.0).collect().item(),
        df_miss,
        round((df_miss / df_cell) * 100.0, 2),
        
    ]}, strict=False)

summary

metric,before,after
str,str,str
"""Rows""","""2925493""","""994341"""
"""Columns""","""142""","""89"""
"""Date range""","""2007-06-01""","""2016-01-01"""
"""Default rate""","""12.43""","""20.74"""
"""Missing cells""","""107893231""","""4261315"""
"""Missing cell rate""","""25.97""","""4.82"""
